# Advanced Problems with Solutions: Reversed Iteration

This notebook expands the core idea of **reversed iteration** in Python:

- `reversed(obj)` first looks for `obj.__reversed__()`.
- If `__reversed__` is missing, Python can fall back to the sequence protocol when `__len__` and `__getitem__` are available.
- General iterators and generators are not automatically reversible because they may be infinite, stateful, or already exhausted.

Each problem includes a complete solution and runnable checks.

## Best-practice checklist

Use this checklist while solving the problems:

1. Implement `__reversed__` when reverse traversal is natural and should be lazy.
2. Return a **new independent iterator** every time `__iter__` or `__reversed__` is called.
3. Use the sequence protocol (`__len__` + `__getitem__`) when random access is natural.
4. Avoid materializing large iterables just to reverse them.
5. Never try to reverse an infinite iterator without bounding it first.
6. Decide whether reverse iteration should be **live** or a **snapshot**.
7. In `__getitem__`, raise `IndexError` for invalid indexes.
8. Add small assertions that prove forward and reverse behavior.

In [1]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from itertools import chain, count, islice
from pathlib import Path
import tempfile

## Problem 1 — Build a lazy sequence that works with `reversed()`

Create a `LazySquares` class that does not precompute a list of squares.

Requirements:

- `LazySquares(n)` represents `0**2, 1**2, ..., (n-1)**2`.
- It must support `len(obj)`.
- It must support indexing and slicing.
- It must work with normal iteration.
- It must work with `reversed(obj)` without defining `__reversed__`.

In [2]:
class LazySquares:
    def __init__(self, n):
        if n < 0:
            raise ValueError("n must be non-negative")
        self._n = n

    def __len__(self):
        return self._n

    def __getitem__(self, index):
        if isinstance(index, slice):
            return [self[i] for i in range(*index.indices(self._n))]

        if index < 0:
            index += self._n

        if index < 0 or index >= self._n:
            raise IndexError(index)

        return index ** 2


sq = LazySquares(6)

assert len(sq) == 6
assert list(sq) == [0, 1, 4, 9, 16, 25]
assert list(reversed(sq)) == [25, 16, 9, 4, 1, 0]
assert sq[2:5] == [4, 9, 16]
assert sq[-1] == 25

print("Problem 1 passed")

Problem 1 passed


## Problem 2 — Implement explicit reverse iteration for a lazy card deck

Create a deck that does not store all 52 cards in a list.

Requirements:

- The deck has 4 suits and 13 ranks.
- Forward iteration starts at `2 of Spades`.
- Reverse iteration starts at `Ace of Clubs`.
- `__reversed__` must be lazy.
- The same deck instance must be reusable.

In [3]:
_SUITS = ("Spades", "Hearts", "Diamonds", "Clubs")
_RANKS = tuple(range(2, 11)) + tuple("JQKA")


@dataclass(frozen=True)
class Card:
    rank: object
    suit: str


class CardDeck:
    def __len__(self):
        return len(_SUITS) * len(_RANKS)

    def _card_at(self, index):
        if index < 0:
            index += len(self)
        if index < 0 or index >= len(self):
            raise IndexError(index)

        suit = _SUITS[index // len(_RANKS)]
        rank = _RANKS[index % len(_RANKS)]
        return Card(rank, suit)

    def __iter__(self):
        for index in range(len(self)):
            yield self._card_at(index)

    def __reversed__(self):
        for index in range(len(self) - 1, -1, -1):
            yield self._card_at(index)


deck = CardDeck()

assert len(deck) == 52
assert next(iter(deck)) == Card(2, "Spades")
assert next(reversed(deck)) == Card("A", "Clubs")
assert list(islice(reversed(deck), 7)) == [
    Card("A", "Clubs"),
    Card("K", "Clubs"),
    Card("Q", "Clubs"),
    Card("J", "Clubs"),
    Card(10, "Clubs"),
    Card(9, "Clubs"),
    Card(8, "Clubs"),
]

print("Problem 2 passed")

Problem 2 passed


## Problem 3 — Let the sequence protocol handle `reversed()`

Rewrite the deck as a true sequence.

Requirements:

- Implement `__len__`.
- Implement `__getitem__`.
- Do **not** implement `__iter__`.
- Do **not** implement `__reversed__`.
- Confirm that both `for card in deck` and `reversed(deck)` still work.

In [4]:
class IndexedCardDeck:
    def __len__(self):
        return len(_SUITS) * len(_RANKS)

    def __getitem__(self, index):
        if index < 0:
            index += len(self)

        if index < 0 or index >= len(self):
            raise IndexError(index)

        suit = _SUITS[index // len(_RANKS)]
        rank = _RANKS[index % len(_RANKS)]
        return Card(rank, suit)


indexed_deck = IndexedCardDeck()

assert list(islice(indexed_deck, 3)) == [
    Card(2, "Spades"),
    Card(3, "Spades"),
    Card(4, "Spades"),
]
assert list(islice(reversed(indexed_deck), 3)) == [
    Card("A", "Clubs"),
    Card("K", "Clubs"),
    Card("Q", "Clubs"),
]

print("Problem 3 passed")

Problem 3 passed


## Problem 4 — Override the default reverse behavior

A custom object can be a sequence but still define `__reversed__` to mean something domain-specific.

Create `ScoreBoard`:

- Forward iteration returns players in insertion order.
- Indexing returns players in insertion order.
- `reversed(board)` returns players by descending score, not insertion order.

In [5]:
@dataclass(frozen=True)
class Player:
    name: str
    score: int


class ScoreBoard:
    def __init__(self, players):
        self._players = tuple(players)

    def __len__(self):
        return len(self._players)

    def __getitem__(self, index):
        return self._players[index]

    def __reversed__(self):
        return iter(sorted(self._players, key=lambda player: player.score, reverse=True))


board = ScoreBoard([
    Player("Ada", 20),
    Player("Grace", 35),
    Player("Linus", 15),
])

assert [p.name for p in board] == ["Ada", "Grace", "Linus"]
assert [p.name for p in reversed(board)] == ["Grace", "Ada", "Linus"]

print("Problem 4 passed")

Problem 4 passed


## Problem 5 — Never return an exhausted iterator from `__reversed__`

A common bug is to store one reverse iterator and return it repeatedly.

Demonstrate the bug, then fix it.

In [6]:
class BrokenReverse:
    def __init__(self, items):
        self._items = list(items)
        self._reverse_iterator = reversed(self._items)

    def __iter__(self):
        return iter(self._items)

    def __reversed__(self):
        return self._reverse_iterator


broken = BrokenReverse([1, 2, 3])

assert list(reversed(broken)) == [3, 2, 1]
assert list(reversed(broken)) == []  # Bug: the stored iterator is exhausted


class FixedReverse:
    def __init__(self, items):
        self._items = list(items)

    def __iter__(self):
        return iter(self._items)

    def __reversed__(self):
        return reversed(self._items)


fixed = FixedReverse([1, 2, 3])

assert list(reversed(fixed)) == [3, 2, 1]
assert list(reversed(fixed)) == [3, 2, 1]

print("Problem 5 passed")

Problem 5 passed


## Problem 6 — Choose snapshot semantics for safer reverse iteration

Create a history object whose reverse iterator is a snapshot.

Requirement:

- If the object changes after `reversed(history)` is called, the active reverse iterator should not see the new event.

In [7]:
class EventHistory:
    def __init__(self):
        self._events = []

    def append(self, event):
        self._events.append(event)

    def __iter__(self):
        return iter(tuple(self._events))

    def __reversed__(self):
        return reversed(tuple(self._events))


history = EventHistory()
history.append("login")
history.append("open-dashboard")
history.append("export-report")

reverse_iterator = reversed(history)

history.append("logout")

assert list(reverse_iterator) == ["export-report", "open-dashboard", "login"]
assert list(reversed(history)) == ["logout", "export-report", "open-dashboard", "login"]

print("Problem 6 passed")

Problem 6 passed


## Problem 7 — Reverse a finite generator safely

Generators are not reversible. Build a helper that materializes only when a maximum size is provided.

Requirements:

- `safe_reversed(iterable)` should use `reversed(iterable)` when possible.
- If the object is not reversible, require `max_materialize`.
- If materialization exceeds the limit, raise `ValueError`.

In [8]:
def safe_reversed(iterable, *, max_materialize=None):
    try:
        return reversed(iterable)
    except TypeError:
        if max_materialize is None:
            raise TypeError(
                "This iterable is not reversible. Provide max_materialize "
                "to explicitly allow bounded materialization."
            )

        items = list(islice(iterable, max_materialize + 1))

        if len(items) > max_materialize:
            raise ValueError("Materialization limit exceeded")

        return reversed(items)


assert list(safe_reversed([1, 2, 3])) == [3, 2, 1]
assert list(safe_reversed((x * 10 for x in range(4)), max_materialize=4)) == [30, 20, 10, 0]

try:
    list(safe_reversed((x for x in range(10)), max_materialize=5))
except ValueError as ex:
    assert "limit exceeded" in str(ex)
else:
    raise AssertionError("Expected ValueError")

print("Problem 7 passed")

Problem 7 passed


## Problem 8 — Reverse only the last `n` items of a stream

A stream may be huge or infinite. You cannot reverse the entire stream.

Create `last_n_reversed(iterable, n)`.

Requirements:

- Keep only the last `n` items.
- Return them newest-to-oldest.
- Work with generators.

In [9]:
def last_n_reversed(iterable, n):
    if n < 0:
        raise ValueError("n must be non-negative")
    return reversed(deque(iterable, maxlen=n))


stream = (x for x in range(100))
assert list(last_n_reversed(stream, 5)) == [99, 98, 97, 96, 95]

bounded_infinite_stream = islice(count(10), 8)
assert list(last_n_reversed(bounded_infinite_stream, 3)) == [17, 16, 15]

print("Problem 8 passed")

Problem 8 passed


## Problem 9 — Read a text file backwards without loading the whole file

Implement a chunked reverse line reader.

Requirements:

- Read bytes from the end of the file in chunks.
- Yield text lines from bottom to top.
- Do not call `readlines()` or load the full file into a list.

In [10]:
def reverse_readlines(path, *, chunk_size=64, encoding="utf-8"):
    """Yield lines from a text file in reverse order, without loading the file."""
    path = Path(path)

    def decode_line(raw_line):
        return raw_line.decode(encoding).removesuffix("\r")

    with path.open("rb") as file:
        file.seek(0, 2)
        position = file.tell()
        buffer = b""
        first_line_seen = False

        while position > 0:
            read_size = min(chunk_size, position)
            position -= read_size

            file.seek(position)
            chunk = file.read(read_size)
            buffer = chunk + buffer

            parts = buffer.split(b"\n")
            buffer = parts[0]

            for raw_line in reversed(parts[1:]):
                if not first_line_seen and raw_line == b"":
                    first_line_seen = True
                    continue

                first_line_seen = True
                yield decode_line(raw_line)

        if buffer:
            yield decode_line(buffer)


with tempfile.NamedTemporaryFile("w+", delete=False, encoding="utf-8") as temp:
    temp.write("alpha\nbeta\ngamma\ndelta\n")
    temp_path = temp.name

try:
    assert list(reverse_readlines(temp_path, chunk_size=5)) == [
        "delta",
        "gamma",
        "beta",
        "alpha",
    ]
finally:
    Path(temp_path).unlink(missing_ok=True)

print("Problem 9 passed")

Problem 9 passed


## Problem 10 — Reverse dictionary views intentionally

Modern Python dictionaries preserve insertion order, and dictionary views are reversible.

Create utility functions:

- `newest_keys(mapping)`
- `newest_values(mapping)`
- `newest_items(mapping)`

Each should return newest-to-oldest data based on insertion order.

In [11]:
def newest_keys(mapping):
    return list(reversed(mapping.keys()))


def newest_values(mapping):
    return list(reversed(mapping.values()))


def newest_items(mapping):
    return list(reversed(mapping.items()))


config_versions = {
    "v1": "baseline",
    "v2": "cache-added",
    "v3": "rate-limited",
}

assert newest_keys(config_versions) == ["v3", "v2", "v1"]
assert newest_values(config_versions) == ["rate-limited", "cache-added", "baseline"]
assert newest_items(config_versions) == [
    ("v3", "rate-limited"),
    ("v2", "cache-added"),
    ("v1", "baseline"),
]

print("Problem 10 passed")

Problem 10 passed


## Problem 11 — Reverse a paginated collection lazily

Simulate an API that returns pages of events.

Requirements:

- Forward iteration fetches page 1, then page 2, etc.
- Reverse iteration fetches the last page first.
- Items inside each reverse page are also yielded from newest to oldest.
- Track the order in which pages were fetched.

In [12]:
class FakeEventAPI:
    def __init__(self, pages):
        self._pages = [tuple(page) for page in pages]
        self.fetch_log = []

    @property
    def page_count(self):
        return len(self._pages)

    def fetch_page(self, page_number):
        if page_number < 1 or page_number > self.page_count:
            raise IndexError(page_number)

        self.fetch_log.append(page_number)
        return self._pages[page_number - 1]


class PaginatedEvents:
    def __init__(self, api):
        self._api = api

    def __iter__(self):
        for page_number in range(1, self._api.page_count + 1):
            yield from self._api.fetch_page(page_number)

    def __reversed__(self):
        for page_number in range(self._api.page_count, 0, -1):
            yield from reversed(self._api.fetch_page(page_number))


api = FakeEventAPI([
    ["e1", "e2"],
    ["e3", "e4"],
    ["e5"],
])
events = PaginatedEvents(api)

assert list(events) == ["e1", "e2", "e3", "e4", "e5"]
assert api.fetch_log == [1, 2, 3]

api.fetch_log.clear()

assert list(reversed(events)) == ["e5", "e4", "e3", "e2", "e1"]
assert api.fetch_log == [3, 2, 1]

print("Problem 11 passed")

Problem 11 passed


## Problem 12 — Reverse a fixed-size history buffer

Create a ring-buffer-like history object.

Requirements:

- Keep only the last `maxlen` records.
- Forward iteration yields oldest-to-newest.
- Reverse iteration yields newest-to-oldest.

In [13]:
class RecentHistory:
    def __init__(self, maxlen):
        if maxlen <= 0:
            raise ValueError("maxlen must be positive")
        self._items = deque(maxlen=maxlen)

    def append(self, item):
        self._items.append(item)

    def __iter__(self):
        return iter(self._items)

    def __reversed__(self):
        return reversed(self._items)


recent = RecentHistory(maxlen=3)

for event in ["build-1", "build-2", "build-3", "build-4"]:
    recent.append(event)

assert list(recent) == ["build-2", "build-3", "build-4"]
assert list(reversed(recent)) == ["build-4", "build-3", "build-2"]

print("Problem 12 passed")

Problem 12 passed


## Problem 13 — Reverse rows and reverse cells in a matrix

Create a matrix with two different reverse views.

Requirements:

- `for row in matrix` yields rows top-to-bottom.
- `reversed(matrix)` yields rows bottom-to-top.
- `matrix.reverse_scan()` yields cells from bottom-right to top-left.

In [14]:
class Matrix:
    def __init__(self, rows):
        self._rows = tuple(tuple(row) for row in rows)

        if self._rows:
            width = len(self._rows[0])
            if any(len(row) != width for row in self._rows):
                raise ValueError("All rows must have the same length")

    def __iter__(self):
        return iter(self._rows)

    def __reversed__(self):
        return reversed(self._rows)

    def reverse_scan(self):
        for row in reversed(self._rows):
            yield from reversed(row)


matrix = Matrix([
    [1, 2, 3],
    [4, 5, 6],
])

assert list(matrix) == [(1, 2, 3), (4, 5, 6)]
assert list(reversed(matrix)) == [(4, 5, 6), (1, 2, 3)]
assert list(matrix.reverse_scan()) == [6, 5, 4, 3, 2, 1]

print("Problem 13 passed")

Problem 13 passed


## Problem 14 — Add reverse iteration to a singly linked list

A singly linked list cannot naturally move backwards. A practical reverse iterator may need temporary storage.

Requirements:

- Forward iteration walks nodes from head to tail.
- Reverse iteration may use a stack/list internally.
- The public linked list should not expose its nodes.

In [15]:
@dataclass
class _Node:
    value: object
    next: "_Node | None" = None


class SinglyLinkedList:
    def __init__(self, values=()):
        self._head = None
        self._tail = None

        for value in values:
            self.append(value)

    def append(self, value):
        node = _Node(value)

        if self._head is None:
            self._head = self._tail = node
        else:
            self._tail.next = node
            self._tail = node

    def __iter__(self):
        current = self._head

        while current is not None:
            yield current.value
            current = current.next

    def __reversed__(self):
        return reversed(list(self))


numbers = SinglyLinkedList([10, 20, 30, 40])

assert list(numbers) == [10, 20, 30, 40]
assert list(reversed(numbers)) == [40, 30, 20, 10]

print("Problem 14 passed")

Problem 14 passed


## Problem 15 — Reverse traversal of a binary search tree

Create a binary search tree.

Requirements:

- Forward iteration yields values in sorted ascending order.
- Reverse iteration yields values in sorted descending order.
- Traversal must be lazy.

In [16]:
@dataclass
class _BSTNode:
    value: int
    left: "_BSTNode | None" = None
    right: "_BSTNode | None" = None

    def insert(self, value):
        if value < self.value:
            if self.left is None:
                self.left = _BSTNode(value)
            else:
                self.left.insert(value)
        else:
            if self.right is None:
                self.right = _BSTNode(value)
            else:
                self.right.insert(value)


class BinarySearchTree:
    def __init__(self, values=()):
        self._root = None

        for value in values:
            self.insert(value)

    def insert(self, value):
        if self._root is None:
            self._root = _BSTNode(value)
        else:
            self._root.insert(value)

    def _walk_forward(self, node):
        if node is not None:
            yield from self._walk_forward(node.left)
            yield node.value
            yield from self._walk_forward(node.right)

    def _walk_reverse(self, node):
        if node is not None:
            yield from self._walk_reverse(node.right)
            yield node.value
            yield from self._walk_reverse(node.left)

    def __iter__(self):
        yield from self._walk_forward(self._root)

    def __reversed__(self):
        yield from self._walk_reverse(self._root)


tree = BinarySearchTree([7, 3, 10, 1, 5, 9, 12])

assert list(tree) == [1, 3, 5, 7, 9, 10, 12]
assert list(reversed(tree)) == [12, 10, 9, 7, 5, 3, 1]

print("Problem 15 passed")

Problem 15 passed


## Problem 16 — Build a reusable range-like object

Use Python's built-in `range` internally, but expose your own iterable.

Requirements:

- Forward iteration delegates to `range`.
- Reverse iteration delegates to `reversed(range(...))`.
- The object must work with positive and negative steps.

In [17]:
class RangeView:
    def __init__(self, start, stop=None, step=1):
        if stop is None:
            start, stop = 0, start

        self._range = range(start, stop, step)

    def __len__(self):
        return len(self._range)

    def __iter__(self):
        return iter(self._range)

    def __reversed__(self):
        return reversed(self._range)


assert list(RangeView(5)) == [0, 1, 2, 3, 4]
assert list(reversed(RangeView(5))) == [4, 3, 2, 1, 0]

assert list(RangeView(10, 0, -3)) == [10, 7, 4, 1]
assert list(reversed(RangeView(10, 0, -3))) == [1, 4, 7, 10]

print("Problem 16 passed")

Problem 16 passed


## Problem 17 — Create multiple reverse views for a task board

Sometimes `reversed(obj)` is not enough. Add explicit named views.

Create `TaskBoard`.

Requirements:

- Default iteration returns tasks in creation order.
- `reversed(board)` returns newest-to-oldest.
- `board.by_priority_desc()` returns highest-priority first.
- Keep the internal task storage private.

In [18]:
@dataclass(frozen=True)
class Task:
    title: str
    priority: int


class TaskBoard:
    def __init__(self):
        self._tasks = []

    def add(self, title, priority):
        self._tasks.append(Task(title, priority))

    def __iter__(self):
        return iter(tuple(self._tasks))

    def __reversed__(self):
        return reversed(tuple(self._tasks))

    def by_priority_desc(self):
        return iter(sorted(self._tasks, key=lambda task: task.priority, reverse=True))


tasks = TaskBoard()
tasks.add("write docs", 2)
tasks.add("fix production bug", 5)
tasks.add("refactor", 3)

assert [task.title for task in tasks] == [
    "write docs",
    "fix production bug",
    "refactor",
]
assert [task.title for task in reversed(tasks)] == [
    "refactor",
    "fix production bug",
    "write docs",
]
assert [task.title for task in tasks.by_priority_desc()] == [
    "fix production bug",
    "refactor",
    "write docs",
]

print("Problem 17 passed")

Problem 17 passed


## Problem 18 — Reverse grouped records with `itertools.chain`

You have grouped records by day. Within each day, records are chronological.

Requirements:

- Forward iteration yields Monday records, then Tuesday records, etc.
- Reverse iteration yields the latest day first.
- Within each day, reverse iteration also yields newest-to-oldest.
- Use delegation where possible.

In [19]:
class GroupedRecords:
    def __init__(self, groups):
        self._groups = tuple(tuple(group) for group in groups)

    def __iter__(self):
        return chain.from_iterable(self._groups)

    def __reversed__(self):
        reverse_groups = (reversed(group) for group in reversed(self._groups))
        return chain.from_iterable(reverse_groups)


records = GroupedRecords([
    ["mon-09:00", "mon-10:00"],
    ["tue-08:00"],
    ["wed-11:00", "wed-12:00"],
])

assert list(records) == [
    "mon-09:00",
    "mon-10:00",
    "tue-08:00",
    "wed-11:00",
    "wed-12:00",
]
assert list(reversed(records)) == [
    "wed-12:00",
    "wed-11:00",
    "tue-08:00",
    "mon-10:00",
    "mon-09:00",
]

print("Problem 18 passed")

Problem 18 passed


## Problem 19 — Build a `ReversedView` adapter

Create a small adapter that gives a reusable reverse view over an existing reversible object.

Requirements:

- `ReversedView([1, 2, 3])` should iterate as `3, 2, 1`.
- Calling `reversed()` on the view should return the original order.
- Do not copy the underlying object.

In [20]:
class ReversedView:
    def __init__(self, reversible):
        # Validate early so errors are easier to understand.
        try:
            reversed(reversible)
        except TypeError as ex:
            raise TypeError("Object must be reversible") from ex

        self._reversible = reversible

    def __iter__(self):
        return reversed(self._reversible)

    def __reversed__(self):
        return iter(self._reversible)


data = [1, 2, 3, 4]
view = ReversedView(data)

assert list(view) == [4, 3, 2, 1]
assert list(reversed(view)) == [1, 2, 3, 4]

data.append(5)

assert list(view) == [5, 4, 3, 2, 1]
assert list(reversed(view)) == [1, 2, 3, 4, 5]

print("Problem 19 passed")

Problem 19 passed


## Problem 20 — Capstone: build a reversible playlist

Create a `Playlist` class.

Requirements:

- Store tracks internally as private data.
- Support `len(playlist)`.
- Support indexing and slicing.
- Forward iteration returns tracks in playlist order.
- Reverse iteration returns tracks from last to first.
- `recent(n)` returns the last `n` tracks newest-to-oldest.
- Active iterators should use snapshot semantics.

In [21]:
@dataclass(frozen=True)
class Track:
    artist: str
    title: str


class Playlist:
    def __init__(self, tracks=()):
        self._tracks = list(tracks)

    def add(self, artist, title):
        self._tracks.append(Track(artist, title))

    def __len__(self):
        return len(self._tracks)

    def __getitem__(self, index):
        if isinstance(index, slice):
            return tuple(self._tracks[index])
        return self._tracks[index]

    def __iter__(self):
        return iter(tuple(self._tracks))

    def __reversed__(self):
        return reversed(tuple(self._tracks))

    def recent(self, n):
        if n < 0:
            raise ValueError("n must be non-negative")
        return islice(reversed(self), n)


playlist = Playlist()
playlist.add("Massive Attack", "Teardrop")
playlist.add("Portishead", "Roads")
playlist.add("Radiohead", "Everything in Its Right Place")

assert len(playlist) == 3
assert playlist[0] == Track("Massive Attack", "Teardrop")
assert playlist[-1] == Track("Radiohead", "Everything in Its Right Place")
assert playlist[1:] == (
    Track("Portishead", "Roads"),
    Track("Radiohead", "Everything in Its Right Place"),
)

iterator = iter(playlist)
reverse_iterator = reversed(playlist)

playlist.add("Björk", "Jóga")

assert [track.title for track in iterator] == [
    "Teardrop",
    "Roads",
    "Everything in Its Right Place",
]
assert [track.title for track in reverse_iterator] == [
    "Everything in Its Right Place",
    "Roads",
    "Teardrop",
]
assert [track.title for track in playlist.recent(2)] == [
    "Jóga",
    "Everything in Its Right Place",
]

print("Problem 20 passed")

Problem 20 passed


## Final review questions

Answer these after completing the notebook:

1. Why can `reversed([1, 2, 3])` work but `reversed(x for x in range(3))` fail?
2. What does Python try before using the sequence fallback?
3. Why should `__reversed__` return a new iterator every time?
4. When is it better to implement `__getitem__` instead of `__reversed__`?
5. What is the difference between live reverse iteration and snapshot reverse iteration?
6. Why is reversing an infinite iterable impossible without a bound?
7. Why should invalid indexes in `__getitem__` raise `IndexError`?

## Summary

You practiced reversed iteration using:

- explicit `__reversed__`
- the sequence protocol
- lazy reverse traversal
- bounded materialization
- snapshot semantics
- dictionaries and views
- files, streams, pagination, linked lists, trees, and custom containers

The key design idea is simple: reverse iteration is not just about flipping order; it is about choosing the right behavior, cost model, and safety guarantees for your object.